# Trainer comparison notebook

This notebook validates and compares four real trainer paths:

- `PrioritySearch` as the standard Trace baseline
- `TextGradTrainer`
- `OpenEvolveTrainer`
- `DSPyTrainer`

It checks out `textgrad_openevolve`, installs real optional packages when needed, runs focused integration checks, and runs a small real optimization demo with OpenRouter or OpenAI. The DSPy row uses a DSPy-native task; the Trace/TextGrad/OpenEvolve rows use a Trace scalar task.

## What this notebook verifies

- required trainer packages import from real installations
- Trace-Bench discovers the trainer classes
- focused trainer tests and compile checks pass
- every comparison row learns from three examples and reports three held-out examples
- result tables show before/after scores, per-example outputs, and red highlighting for rows with no held-out improvement

## High-level interpretation guide

Use the notebook in this order:

1. Confirm the branch, package imports, trainer discovery, and focused tests.
2. Compare before/after train and held-out scores.
3. Treat any red row as a real trainer run that completed or failed without held-out improvement.

In [1]:
import os
import sys
import subprocess
from collections.abc import Sequence
from pathlib import Path
from subprocess import CompletedProcess

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
CURRENT_REPO = Path.cwd()
TRACE_BENCH_REMOTE_URL = "https://github.com/doxav/Trace-Bench.git"
TRACE_BENCH_BRANCH = "textgrad_openevolve"
TRACE_BENCH_REPO = CURRENT_REPO if (CURRENT_REPO / "trace_bench").is_dir() else WORKDIR / "Trace-Bench"
NEWTRACE_REMOTE_URL = "https://github.com/doxav/NewTrace.git"
NEWTRACE_BRANCH = "experimental"
NEWTRACE_REPO = WORKDIR / "NewTrace"
OPENEVOLVE_REMOTE_URL = "https://github.com/algorithmicsuperintelligence/openevolve.git"
OPENEVOLVE_BRANCH = "main"
OPENEVOLVE_REPO = WORKDIR / "openevolve"

for repo_path in (NEWTRACE_REPO, TRACE_BENCH_REPO):
    repo_path_str = str(repo_path)
    if repo_path_str not in sys.path:
        sys.path.insert(0, repo_path_str)

def run(cmd: Sequence[str | os.PathLike[str]], cwd: Path | str | None = None, check: bool = True) -> CompletedProcess[bytes]:
    """Run a subprocess command and echo its argv without shell interpolation."""
    print("$", " ".join(map(str, cmd)))
    return subprocess.run([str(part) for part in cmd], cwd=cwd, check=check)

def checkout_branch(repo_path: Path, remote_url: str, branch: str) -> None:
    """Fetch, checkout, and fast-forward a branch in an existing clone."""
    run(["git", "fetch", remote_url, branch], cwd=repo_path)
    checkout = run(["git", "checkout", branch], cwd=repo_path, check=False)
    if checkout.returncode != 0:
        run(["git", "checkout", "-b", branch, "FETCH_HEAD"], cwd=repo_path)
    run(["git", "pull", "--ff-only", remote_url, branch], cwd=repo_path)

print("WORKDIR =", WORKDIR)
print("TRACE_BENCH_REMOTE_URL =", TRACE_BENCH_REMOTE_URL)
print("TRACE_BENCH_BRANCH =", TRACE_BENCH_BRANCH)
print("TRACE_BENCH_REPO =", TRACE_BENCH_REPO)
print("NEWTRACE_REMOTE_URL =", NEWTRACE_REMOTE_URL)
print("NEWTRACE_BRANCH =", NEWTRACE_BRANCH)
print("NEWTRACE_REPO =", NEWTRACE_REPO)
print("OPENEVOLVE_REMOTE_URL =", OPENEVOLVE_REMOTE_URL)
print("OPENEVOLVE_BRANCH =", OPENEVOLVE_BRANCH)
print("OPENEVOLVE_REPO =", OPENEVOLVE_REPO)

WORKDIR = /home/xav/code/Trace-Bench
TRACE_BENCH_REMOTE_URL = https://github.com/doxav/Trace-Bench.git
TRACE_BENCH_BRANCH = textgrad_openevolve
TRACE_BENCH_REPO = /home/xav/code/Trace-Bench
NEWTRACE_REMOTE_URL = https://github.com/doxav/NewTrace.git
NEWTRACE_BRANCH = experimental
NEWTRACE_REPO = /home/xav/code/Trace-Bench/NewTrace
OPENEVOLVE_REMOTE_URL = https://github.com/algorithmicsuperintelligence/openevolve.git
OPENEVOLVE_BRANCH = main
OPENEVOLVE_REPO = /home/xav/code/Trace-Bench/openevolve


## 1. Clone and checkout the repositories

This clones:
- `Trace-Bench` on `textgrad_openevolve`
- `doxav/NewTrace` on `experimental`
- `OpenEvolve` only if the real package is missing

Skip this if you already have local checkouts and want to point the notebook at them manually.

In [2]:
if not TRACE_BENCH_REPO.exists():
    run([
        "git", "clone",
        "--branch", TRACE_BENCH_BRANCH,
        "--single-branch",
        TRACE_BENCH_REMOTE_URL,
        str(TRACE_BENCH_REPO),
    ])
elif TRACE_BENCH_REPO.resolve() == CURRENT_REPO.resolve():
    branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=TRACE_BENCH_REPO, text=True).strip()
    if branch != TRACE_BENCH_BRANCH:
        checkout_branch(TRACE_BENCH_REPO, TRACE_BENCH_REMOTE_URL, TRACE_BENCH_BRANCH)
    else:
        print(f"Trace-Bench current checkout is already on {TRACE_BENCH_BRANCH}; preserving local edits.")
else:
    print(f"Trace-Bench already exists; checking out {TRACE_BENCH_BRANCH}.")
    checkout_branch(TRACE_BENCH_REPO, TRACE_BENCH_REMOTE_URL, TRACE_BENCH_BRANCH)

if not NEWTRACE_REPO.exists():
    run([
        "git", "clone",
        "--branch", NEWTRACE_BRANCH,
        "--single-branch",
        NEWTRACE_REMOTE_URL,
        str(NEWTRACE_REPO),
    ])
else:
    print(f"NewTrace already exists; checking out {NEWTRACE_BRANCH}.")
    checkout_branch(NEWTRACE_REPO, NEWTRACE_REMOTE_URL, NEWTRACE_BRANCH)

Trace-Bench current checkout is already on textgrad_openevolve; preserving local edits.
$ git clone --branch experimental --single-branch https://github.com/doxav/NewTrace.git /home/xav/code/Trace-Bench/NewTrace


Cloning into '/home/xav/code/Trace-Bench/NewTrace'...


## 2. Install Python dependencies

This installs NewTrace and Trace-Bench editable, plus the real optional trainer packages used by the demo. If `openevolve.run_evolution` is not importable, OpenEvolve is cloned and installed editable.

In [3]:
run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q",
     "graphviz", "pyyaml", "pytest", "litellm", "aiohttp", "nest_asyncio", "dspy-ai", "optuna",
     "tensorboard", "tensorboardX", "scikit-learn", "datasets", "openai", "pandas"])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(NEWTRACE_REPO)])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRACE_BENCH_REPO)])

def has_real_openevolve() -> bool:
    """Return True only when the real OpenEvolve API is importable."""
    try:
        import openevolve
        return callable(getattr(openevolve, "run_evolution", None))
    except Exception:
        return False

if not has_real_openevolve():
    if not OPENEVOLVE_REPO.exists():
        run([
            "git", "clone",
            "--branch", OPENEVOLVE_BRANCH,
            "--single-branch",
            OPENEVOLVE_REMOTE_URL,
            str(OPENEVOLVE_REPO),
        ])
    else:
        print(f"OpenEvolve already exists; checking out {OPENEVOLVE_BRANCH}.")
        checkout_branch(OPENEVOLVE_REPO, OPENEVOLVE_REMOTE_URL, OPENEVOLVE_BRANCH)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", str(OPENEVOLVE_REPO)])

if not has_real_openevolve():
    raise ImportError("OpenEvolve is required for this demo and could not be installed.")

$ /home/xav/miniconda3/bin/python -m pip install -q -U pip setuptools wheel


$ /home/xav/miniconda3/bin/python -m pip install -q graphviz pyyaml pytest litellm aiohttp nest_asyncio dspy-ai optuna tensorboard tensorboardX scikit-learn datasets openai pandas


$ /home/xav/miniconda3/bin/python -m pip install -q -e /home/xav/code/Trace-Bench/NewTrace


$ /home/xav/miniconda3/bin/python -m pip install -q -e /home/xav/code/Trace-Bench


## 3. Provider setup for real online experiments

The comparison requires a real provider. In Colab the cell reads `OPENROUTER_API_KEY` or `OPENAI_API_KEY` from Colab Secrets when present; locally it reads the same environment variables.

In [4]:
from getpass import getpass

def colab_secret(name: str) -> str:
    """Return a Colab Secret value when available, otherwise an empty string."""
    try:
        from google.colab import userdata
    except Exception:
        return ""
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

PROVIDER = "auto"  # @param ["auto", "openrouter", "openai", "none"]
MODEL = ""  # @param {type:"string"}

openrouter_key = os.environ.get("OPENROUTER_API_KEY") or colab_secret("OPENROUTER_API_KEY")
openai_key = os.environ.get("OPENAI_API_KEY") or colab_secret("OPENAI_API_KEY")
MODEL = MODEL or os.environ.get("TRACE_LITELLM_MODEL") or colab_secret("TRACE_LITELLM_MODEL")

if PROVIDER == "auto":
    active_provider = "openrouter" if openrouter_key else "openai" if openai_key else "none"
else:
    active_provider = PROVIDER

if active_provider == "openrouter":
    if not MODEL:
        MODEL = "openrouter/openai/gpt-4o-mini"
    if not openrouter_key:
        openrouter_key = getpass("OPENROUTER_API_KEY: ")
    if not openrouter_key:
        raise ValueError("OPENROUTER_API_KEY is required when PROVIDER is openrouter.")
    os.environ["OPENROUTER_API_KEY"] = openrouter_key
    os.environ["OPENAI_API_KEY"] = openrouter_key
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
elif active_provider == "openai":
    if not MODEL:
        MODEL = "gpt-4o-mini"
    if not openai_key:
        openai_key = getpass("OPENAI_API_KEY: ")
    if not openai_key:
        raise ValueError("OPENAI_API_KEY is required when PROVIDER is openai.")
    os.environ["OPENAI_API_KEY"] = openai_key
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
elif active_provider == "none":
    print("Skipping online provider configuration.")
else:
    raise ValueError(f"Unsupported PROVIDER: {PROVIDER}")

print("PROVIDER =", active_provider)
print("TRACE_LITELLM_MODEL =", os.environ.get("TRACE_LITELLM_MODEL"))
print("OPENAI_BASE_URL =", os.environ.get("OPENAI_BASE_URL"))
print("OPENROUTER_API_KEY configured =", bool(os.environ.get("OPENROUTER_API_KEY")))

PROVIDER = openrouter
TRACE_LITELLM_MODEL = openrouter/openai/gpt-4o-mini
OPENAI_BASE_URL = https://openrouter.ai/api/v1
OPENROUTER_API_KEY configured = True


## 4. Sanity checks and imports

In [5]:
import importlib
import pandas as pd

def required_import(name: str) -> object:
    """Import a required module and raise a descriptive error when unavailable."""
    try:
        module = importlib.import_module(name)
        print("OK:", name)
        return module
    except Exception as exc:
        raise ImportError(f"Required module is unavailable: {name}") from exc

textgrad_module = required_import("opto.optimizers.textgrad")
openevolve_module = required_import("openevolve")
dspy_module = required_import("dspy")
required_import("trace_bench")
required_import("trace_bench.runner")
required_import("trace_bench.registry")
required_import("trace_bench.config")
required_import("trace_bench.trainers.textgrad_trainer")
required_import("trace_bench.trainers.openevolve_trainer")
required_import("trace_bench.trainers.dspy_trainer")

if not callable(getattr(textgrad_module, "TextGrad", None)):
    raise ImportError("opto.optimizers.textgrad.TextGrad is required for this demo.")
if not callable(getattr(openevolve_module, "run_evolution", None)):
    raise ImportError("openevolve.run_evolution is required for this demo.")
if not callable(getattr(dspy_module, "LM", None)):
    raise ImportError("dspy.LM is required for this demo.")

print("TextGrad module:", getattr(textgrad_module, "__file__", "unknown"))
print("OpenEvolve module:", getattr(openevolve_module, "__file__", "unknown"))
print("DSPy module:", getattr(dspy_module, "__file__", "unknown"))

OK: opto.optimizers.textgrad
OK: openevolve


OK: dspy
OK: trace_bench
OK: trace_bench.runner
OK: trace_bench.registry
OK: trace_bench.config
OK: trace_bench.trainers.textgrad_trainer
OK: trace_bench.trainers.openevolve_trainer
OK: trace_bench.trainers.dspy_trainer
TextGrad module: /home/xav/code/Trace-Bench/NewTrace/opto/optimizers/textgrad.py
OpenEvolve module: /home/xav/miniconda3/lib/python3.13/site-packages/openevolve/__init__.py
DSPy module: /home/xav/miniconda3/lib/python3.13/site-packages/dspy/__init__.py


## 5. Focused validation commands

These are the most relevant tests for the new trainers and their integration surface.

In [6]:
TARGETED_TESTS = [
    "tests/test_resolve_external_trainers.py",
    "tests/test_external_utils.py",
    "tests/test_llm_utils.py",
    "tests/test_textgrad_trainer.py",
    "tests/test_openevolve_trainer.py",
    "tests/test_dspy_trainer.py",
]

run([sys.executable, "-m", "pytest", *TARGETED_TESTS, "-q"], cwd=TRACE_BENCH_REPO)
run([sys.executable, "-m", "py_compile",
     "trace_bench/resolve.py",
     "trace_bench/cli.py",
     "trace_bench/runner.py",
     "trace_bench/llm.py",
     "trace_bench/trainers/_external_utils.py",
     "trace_bench/trainers/textgrad_trainer.py",
     "trace_bench/trainers/openevolve_trainer.py",
     "trace_bench/trainers/dspy_trainer.py"], cwd=TRACE_BENCH_REPO)

$ /home/xav/miniconda3/bin/python -m pytest tests/test_resolve_external_trainers.py tests/test_external_utils.py tests/test_llm_utils.py tests/test_textgrad_trainer.py tests/test_openevolve_trainer.py tests/test_dspy_trainer.py -q


................

.                                                        [100%]


17 passed in 2.54s


$ /home/xav/miniconda3/bin/python -m py_compile trace_bench/resolve.py trace_bench/cli.py trace_bench/runner.py trace_bench/llm.py trace_bench/trainers/_external_utils.py trace_bench/trainers/textgrad_trainer.py trace_bench/trainers/openevolve_trainer.py trace_bench/trainers/dspy_trainer.py


CompletedProcess(args=['/home/xav/miniconda3/bin/python', '-m', 'py_compile', 'trace_bench/resolve.py', 'trace_bench/cli.py', 'trace_bench/runner.py', 'trace_bench/llm.py', 'trace_bench/trainers/_external_utils.py', 'trace_bench/trainers/textgrad_trainer.py', 'trace_bench/trainers/openevolve_trainer.py', 'trace_bench/trainers/dspy_trainer.py'], returncode=0)

## 6. Trainer discovery and signatures

This is the fastest way to see whether the branch contains the trainer code and wires it into discovery.

In [7]:
from trace_bench.registry import discover_trainers
from trace_bench.runner import _resolve_algorithm

trainer_rows = []
for spec in discover_trainers():
    if spec.id in {"PrioritySearch", "TextGradTrainer", "OpenEvolveTrainer", "DSPyTrainer"}:
        resolved = _resolve_algorithm(spec.id)
        trainer_rows.append({
            "trainer_id": spec.id,
            "available": spec.available,
            "source": spec.source,
            "resolved_type": type(resolved).__name__ if not isinstance(resolved, type) else "class",
            "resolved_name": getattr(resolved, "__name__", str(resolved)),
            "uses_trace_optimizer": getattr(resolved, "USES_TRACE_OPTIMIZER", None) if isinstance(resolved, type) else None,
            "framework": getattr(resolved, "FRAMEWORK", None) if isinstance(resolved, type) else None,
        })

pd.DataFrame(trainer_rows).sort_values("trainer_id").reset_index(drop=True)

,trainer_id,available,source,resolved_type,resolved_name,uses_trace_optimizer,framework
0,DSPyTrainer,True,trace_bench.trainers.dspy_trainer,class,DSPyTrainer,False,dspy
1,OpenEvolveTrainer,True,trace_bench.trainers.openevolve_trainer,class,OpenEvolveTrainer,False,NaN
2,PrioritySearch,True,opto.features.priority_search.priority_search,str,PrioritySearch,None,NaN
3,TextGradTrainer,True,trace_bench.trainers.textgrad_trainer,class,TextGradTrainer,False,NaN


## 7. Shared helpers for real train/test optimization

The Trace, TextGrad, and OpenEvolve rows use the same Trace scalar parameter task. The DSPy row uses a small routing-code task where MIPROv2 can optimize instructions from labeled examples within notebook runtime.

In [8]:
import contextlib
import io
import re
from typing import Any, Callable

import dspy
from IPython.display import HTML, display

from trace_bench.config import TrainerConfig
from trace_bench.registry import load_task_bundle
from trace_bench.runner import _train_bundle
from trace_bench.trainers._external_utils import apply_parameter_updates

TRACE_TASK_ID = "trace_examples:opentrace_train_single_node"
TASKS_ROOT = str(TRACE_BENCH_REPO / "LLM4AD" / "benchmark_tasks")
TRACE_INITIAL_VALUE = 0.0
TRACE_TARGET_VALUE = 3.0
TRACE_TRAIN_DATASET = {"inputs": ["train-a", "train-b", "train-c"], "infos": [TRACE_TARGET_VALUE] * 3}
TRACE_TEST_DATASET = {"inputs": ["test-a", "test-b", "test-c"], "infos": [TRACE_TARGET_VALUE] * 3}
DSPY_TRAIN_DATASET = {
    "inputs": ["customer tier scarlet", "customer tier azure", "customer tier emerald"],
    "infos": ["A", "B", "C"],
}
DSPY_TEST_DATASET = {
    "inputs": ["routing code for scarlet ticket", "routing code for azure ticket", "routing code for emerald ticket"],
    "infos": ["A", "B", "C"],
}

class RoutingDSPySignature(dspy.Signature):
    """Return the requested routing code as a single uppercase letter."""
    ticket: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="single uppercase letter")

class RoutingDSPyAgent(dspy.Module):
    """Small DSPy module optimized through DSPyTrainer."""
    def __init__(self) -> None:
        super().__init__()
        self.predict = dspy.Predict(RoutingDSPySignature)

    def forward(self, ticket: str) -> str:
        return self.predict(ticket=ticket).answer

    @classmethod
    def to_examples(cls, inputs: list[Any], infos: list[Any]) -> list[Any]:
        return [
            dspy.Example(ticket=str(ticket), answer=str(code), _task=ticket, _info=code).with_inputs("ticket")
            for ticket, code in zip(inputs, infos)
        ]

class RoutingDSPyGuide:
    """Exact-match routing-code metric for DSPy optimizers."""
    def get_feedback(self, _query: Any, response: Any, reference: Any, **_kwargs: Any) -> tuple[float, str]:
        text = str(getattr(response, "data", response)).strip().upper()
        match = re.search(r"[A-Z]", text)
        prediction = match.group(0) if match else text[:1]
        target = str(reference).strip().upper()
        score = 1.0 if prediction == target else 0.0
        return score, f"expected={target}; response={text}"

    def __call__(self, query: Any, response: Any, reference: Any, **kwargs: Any) -> tuple[float, str]:
        return self.get_feedback(query, response, reference, **kwargs)

def _set_only_scalar_trainable(bundle: dict[str, Any]) -> None:
    param = bundle["param"]
    scalar = getattr(param, "value", None)
    if scalar is None:
        scalar = getattr(param, "guess", None)
    if scalar is None:
        raise AttributeError("Scalar demo task requires param.value or param.guess.")
    for parameter in param.parameters():
        parameter.trainable = parameter is scalar
    apply_parameter_updates({scalar: TRACE_INITIAL_VALUE})

def make_trace_demo_bundle() -> dict[str, Any]:
    bundle = load_task_bundle(TRACE_TASK_ID, TASKS_ROOT)
    _set_only_scalar_trainable(bundle)
    bundle["train_dataset"] = TRACE_TRAIN_DATASET
    bundle["test_dataset"] = TRACE_TEST_DATASET
    bundle.pop("validate_dataset", None)
    bundle["optimizer_kwargs"]["objective"] = f"Set the trainable scalar to exactly {TRACE_TARGET_VALUE}."
    bundle["metadata"]["task_label"] = "Trace scalar"
    return bundle

def make_dspy_lm(max_tokens: int = 200) -> Any:
    model = os.environ.get("TRACE_LITELLM_MODEL") or "gpt-4o-mini"
    if "/" not in model and ("gpt" in model.lower() or model.lower().startswith("o")):
        model = f"openai/{model}"
    lm_kwargs: dict[str, Any] = {"cache": False, "max_tokens": max_tokens}
    api_base = os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE")
    if api_base:
        lm_kwargs["api_base"] = api_base
    return dspy.LM(model=model, **lm_kwargs)

def make_dspy_demo_bundle() -> dict[str, Any]:
    dspy.configure(lm=make_dspy_lm())
    return {
        "param": RoutingDSPyAgent(),
        "guide": RoutingDSPyGuide(),
        "train_dataset": DSPY_TRAIN_DATASET,
        "test_dataset": DSPY_TEST_DATASET,
        "optimizer_kwargs": {},
        "metadata": {"task_label": "DSPy routing code", "framework": "dspy"},
    }

def short_text(value: Any, limit: int = 100) -> str:
    text = str(value)
    return text if len(text) <= limit else text[: limit - 3] + "..."

def snapshot_trainable_value(bundle: dict[str, Any]) -> Any:
    scalar = getattr(bundle["param"], "value", None)
    if scalar is None:
        scalar = getattr(bundle["param"], "guess", None)
    if scalar is not None:
        return getattr(scalar, "data", None)
    predictor = getattr(bundle["param"], "predict", None)
    signature = getattr(predictor, "signature", None)
    return short_text(getattr(signature, "instructions", type(bundle["param"]).__name__))

def task_label(bundle: dict[str, Any]) -> str:
    metadata = bundle.get("metadata", {})
    return str(metadata.get("task_label") or metadata.get("benchmark") or "demo")

def output_value(output: Any) -> Any:
    return short_text(getattr(output, "data", output), limit=140)

def score_guide(guide: Any, task_input: Any, response: Any, task_info: Any) -> tuple[float, str]:
    score, feedback = guide(task_input, response, task_info) if callable(guide) else guide.get_feedback(task_input, response, task_info)
    return float(score), str(feedback)

def score_dataset(bundle: dict[str, Any], dataset: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = dataset.get("inputs") or []
    infos = dataset.get("infos") or dataset.get("info") or []
    if len(inputs) != len(infos):
        raise ValueError("Dataset 'inputs' and 'infos' must have the same length.")
    if not inputs:
        raise ValueError("Dataset must contain at least one example.")
    rows = []
    scores = []
    for task_input, task_info in zip(inputs, infos):
        response = output_value(bundle["param"](task_input))
        score, feedback = score_guide(bundle["guide"], task_input, response, task_info)
        scores.append(score)
        rows.append({"input": task_input, "expected": task_info, "output": response, "score": score, "feedback": feedback})
    return {"mean_score": sum(scores) / len(scores), "rows": rows}

def run_train_bundle(
    trainer_id: str,
    params: dict[str, Any] | None = None,
    mode: str = "real",
    logger: str = "none",
    bundle_factory: Callable[[], dict[str, Any]] = make_trace_demo_bundle,
) -> dict[str, Any]:
    bundle = bundle_factory()
    params = params or {}
    train_dataset = bundle["train_dataset"]
    test_dataset = bundle.get("test_dataset") or train_dataset
    before = {
        "value": snapshot_trainable_value(bundle),
        "train": score_dataset(bundle, train_dataset),
        "test": score_dataset(bundle, test_dataset),
    }
    result = _train_bundle(
        bundle=bundle,
        trainer_spec=TrainerConfig(id=trainer_id, params_variants=[params], logger=logger),
        params=params,
        mode=mode,
    )
    after = {
        "value": snapshot_trainable_value(bundle),
        "train": score_dataset(bundle, train_dataset),
        "test": score_dataset(bundle, test_dataset),
    }
    return {
        "trainer_id": trainer_id,
        "task": task_label(bundle),
        "mode": mode,
        "result": result,
        "before": before,
        "after": after,
        "train_examples": len(train_dataset["inputs"]),
        "test_examples": len(test_dataset["inputs"]),
    }

## 8. Real train/test optimization runs

These runs use the real Trace-Bench trainer entry points and real installed trainer packages. Optimizer logs are captured so the notebook output stays focused on the comparison tables.

In [9]:
if active_provider == "none":
    raise RuntimeError("Real comparison requires OPENROUTER_API_KEY or OPENAI_API_KEY.")

REAL_TRAINERS = [
    ("PrioritySearch", "Trace scalar", {
        "ps_steps": 2,
        "ps_batches": 1,
        "num_candidates": 3,
        "num_proposals": 2,
    }, make_trace_demo_bundle),
    ("TextGradTrainer", "Trace scalar", {
        "num_epochs": 2,
        "batch_size": 1,
        "ensure_improvement": True,
        "improvement_threshold": 1e-9,
        "max_tokens": 1024,
    }, make_trace_demo_bundle),
    ("OpenEvolveTrainer", "Trace scalar", {
        "iterations": 4,
        "population_size": 8,
        "num_islands": 1,
        "seed": 3,
        "ensure_improvement": True,
        "improvement_threshold": 1e-9,
        "verbose": False,
        "model": os.environ.get("TRACE_LITELLM_MODEL"),
        "api_base": os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE"),
        "api_key_env": "OPENAI_API_KEY",
        "max_tokens": 2048,
        "temperature": 0.4,
    }, make_trace_demo_bundle),
    ("DSPyTrainer", "DSPy routing code", {
        "dspy_optimizer": "mipro",
        "dspy_lm": make_dspy_lm(),
        "auto": None,
        "num_candidates": 4,
        "num_trials": 5,
        "max_labeled_demos": 3,
        "max_bootstrapped_demos": 1,
        "num_threads": 1,
        "seed": 7,
        "verbose": False,
    }, make_dspy_demo_bundle),
]

smoke_results = []
for trainer_id, task, params, bundle_factory in REAL_TRAINERS:
    captured = io.StringIO()
    try:
        with contextlib.redirect_stdout(captured), contextlib.redirect_stderr(captured):
            item = run_train_bundle(trainer_id, params=params, mode="real", bundle_factory=bundle_factory)
        item["captured_log_tail"] = "\n".join(captured.getvalue().splitlines()[-8:])
        smoke_results.append(item)
        print(f"{trainer_id}: {item['result'].get('status')} ({item['result'].get('resolved_optimizer')})")
    except Exception as exc:
        smoke_results.append({
            "trainer_id": trainer_id,
            "task": task,
            "mode": "real",
            "status": "error",
            "error": f"{type(exc).__name__}: {exc}",
            "captured_log_tail": "\n".join(captured.getvalue().splitlines()[-8:]),
        })
        print(f"{trainer_id}: error")

print(f"Completed {len(smoke_results)} real trainer runs.")

PrioritySearch: ok (OptoPrimeV2)


TextGradTrainer: ok (opto.optimizers.textgrad.TextGrad)


OpenEvolveTrainer: ok (openevolve.run_evolution)


DSPyTrainer: ok (dspy.MIPROv2)
Completed 4 real trainer runs.


In [10]:
summary_rows = []
example_rows = []
for item in smoke_results:
    if "result" not in item:
        summary_rows.append({
            "trainer_id": item["trainer_id"],
            "task": item["task"],
            "mode": item["mode"],
            "status": item["status"],
            "resolved_optimizer": None,
            "before_value": None,
            "after_value": None,
            "train_examples": None,
            "test_examples": None,
            "before_train_score": None,
            "after_train_score": None,
            "train_delta": None,
            "before_test_score": None,
            "after_test_score": None,
            "test_delta": None,
            "improvement": "NO",
            "error": item["error"],
        })
        continue

    result_status = item["result"].get("status")
    before_train = item["before"]["train"]["mean_score"]
    after_train = item["after"]["train"]["mean_score"]
    before_test = item["before"]["test"]["mean_score"]
    after_test = item["after"]["test"]["mean_score"]
    test_delta = after_test - before_test
    improved = result_status == "ok" and test_delta > 0
    summary_rows.append({
        "trainer_id": item["trainer_id"],
        "task": item["task"],
        "mode": item["mode"],
        "status": result_status,
        "resolved_optimizer": item["result"].get("resolved_optimizer"),
        "before_value": item["before"]["value"],
        "after_value": item["after"]["value"],
        "train_examples": item["train_examples"],
        "test_examples": item["test_examples"],
        "before_train_score": before_train,
        "after_train_score": after_train,
        "train_delta": after_train - before_train,
        "before_test_score": before_test,
        "after_test_score": after_test,
        "test_delta": test_delta,
        "improvement": "YES" if improved else "NO",
        "error": item["result"].get("error"),
    })
    for split_name in ("train", "test"):
        for phase in ("before", "after"):
            for index, row in enumerate(item[phase][split_name]["rows"]):
                example_rows.append({
                    "trainer_id": item["trainer_id"],
                    "task": item["task"],
                    "split": split_name,
                    "phase": phase,
                    "example": index,
                    "input": row["input"],
                    "expected": row["expected"],
                    "output": row["output"],
                    "score": row["score"],
                })

trainer_comparison = pd.DataFrame(summary_rows)
example_comparison = pd.DataFrame(example_rows)

score_columns = ["before_train_score", "after_train_score", "train_delta", "before_test_score", "after_test_score", "test_delta"]
def mark_no_improvement(row: pd.Series) -> list[str]:
    style = "background-color: #ffd6d6; color: #9f0000; font-weight: 700"
    return [style if row.get("improvement") != "YES" else "" for _ in row]

styled_comparison = (
    trainer_comparison.style
    .apply(mark_no_improvement, axis=1)
    .format({column: "{:.3f}" for column in score_columns})
)
display(styled_comparison)

no_improvement = trainer_comparison[trainer_comparison["improvement"] != "YES"]
if no_improvement.empty:
    display(HTML("<div style='color:#166534;font-weight:700'>All trainers improved on held-out examples.</div>"))
else:
    names = ", ".join(no_improvement["trainer_id"].astype(str).tolist())
    display(HTML(f"<div style='color:#9f0000;font-weight:800'>NO HELD-OUT IMPROVEMENT: {names}</div>"))

if example_rows:
    display(example_comparison.sort_values(["task", "split", "example", "trainer_id", "phase"]).reset_index(drop=True))
else:
    print("No per-example outputs were produced because all real trainer runs errored.")

for _, row in trainer_comparison.iterrows():
    print(
        f"{row['trainer_id']}: {row['before_test_score']} -> {row['after_test_score']} "
        f"(test_delta={row['test_delta']}, improvement={row['improvement']})"
    )

,trainer_id,task,mode,status,resolved_optimizer,before_value,after_value,train_examples,test_examples,before_train_score,after_train_score,train_delta,before_test_score,after_test_score,test_delta,improvement,error
0,PrioritySearch,Trace scalar,real,ok,OptoPrimeV2,0.000000,3.000000,3,3,-3.000,0.000,3.000,-3.000,0.000,3.000,YES,None
1,TextGradTrainer,Trace scalar,real,ok,opto.optimizers.textgrad.TextGrad,0.000000,3.000000,3,3,-3.000,0.000,3.000,-3.000,0.000,3.000,YES,None
2,OpenEvolveTrainer,Trace scalar,real,ok,openevolve.run_evolution,0.000000,1.000000,3,3,-3.000,-2.000,1.000,-3.000,-2.000,1.000,YES,None
3,DSPyTrainer,DSPy routing code,real,ok,dspy.MIPROv2,Return the requested routing code as a single uppercase letter.,"Based on the customer tier mentioned in the ticket (e.g., ""customer tier scarlet"", ""customer tier...",3,3,0.000,1.000,1.000,0.000,1.000,1.000,YES,None


,trainer_id,task,split,phase,example,input,expected,output,score
0,DSPyTrainer,DSPy routing code,test,after,0,routing code for scarlet ticket,A,A,1.0
1,DSPyTrainer,DSPy routing code,test,before,0,routing code for scarlet ticket,A,S,0.0
2,DSPyTrainer,DSPy routing code,test,after,1,routing code for azure ticket,B,B,1.0
3,DSPyTrainer,DSPy routing code,test,before,1,routing code for azure ticket,B,A,0.0
4,DSPyTrainer,DSPy routing code,test,after,2,routing code for emerald ticket,C,C,1.0
5,DSPyTrainer,DSPy routing code,test,before,2,routing code for emerald ticket,C,E,0.0
6,DSPyTrainer,DSPy routing code,train,after,0,customer tier scarlet,A,A,1.0
7,DSPyTrainer,DSPy routing code,train,before,0,customer tier scarlet,A,S,0.0
8,DSPyTrainer,DSPy routing code,train,after,1,customer tier azure,B,B,1.0
9,DSPyTrainer,DSPy routing code,train,before,1,customer tier azure,B,A,0.0


PrioritySearch: -3.0 -> 0.0 (test_delta=3.0, improvement=YES)
TextGradTrainer: -3.0 -> 0.0 (test_delta=3.0, improvement=YES)
OpenEvolveTrainer: -3.0 -> -2.0 (test_delta=1.0, improvement=YES)
DSPyTrainer: 0.0 -> 1.0 (test_delta=1.0, improvement=YES)


## 9. Practical reading guide

Read red rows first. A red row means the real trainer path either failed or did not improve the held-out score. Then inspect the per-example table to see whether the trainer changed the parameter/instruction and whether the change generalized beyond the three training examples.

## 10. What counts as success

### Strong success
- focused tests pass
- discovery shows the comparison trainers
- real `opto.optimizers.textgrad.TextGrad`, `openevolve.run_evolution`, and `dspy.LM` import successfully
- all real trainer rows complete
- no rows are highlighted red

### Needs follow-up
- a trainer reports an error row
- a trainer completes but is highlighted red because held-out score did not improve
- per-example outputs show memorization or no meaningful parameter/instruction change